# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [97]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [98]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: c:\Users\lenovo\Desktop\hw2\1
FEATURE_DIR: c:\Users\lenovo\Desktop\hw2\1\data\features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [ ]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "")
DB_PORT = int(os.getenv("PGPORT", ""))
DB_NAME = os.getenv("PGDATABASE", "")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [100]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [101]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,core,calendar_day,1,listing_id,bigint,NO
1,core,calendar_day,2,date,date,NO
2,core,calendar_day,3,available,boolean,YES
3,core,calendar_day,4,price,numeric,YES
4,core,calendar_day,5,adjusted_price,numeric,YES
5,core,calendar_day,6,minimum_nights,integer,YES
6,core,calendar_day,7,maximum_nights,integer,YES
7,core,host,1,host_id,bigint,NO
8,core,host,2,host_pseudo_id,text,NO
9,core,host,3,is_superhost,boolean,YES


In [102]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [103]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [104]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684
5,49552,225987,2,Entire home/apt,Entire guest suite,3,2.0,2.0,1 bath,322.0,3,1125,False,0363 576A D827 5085 6B83
6,50263,230246,1,Entire home/apt,Entire condo,4,2.0,3.0,1.5 baths,457.0,2,14,False,0363 7F3D 0BAE 28C8 C7D2
7,50515,231864,14,Entire home/apt,Entire townhouse,5,3.0,3.0,1.5 baths,198.0,7,18,False,0363 5DDB E495 A6D5 CEC6
8,50523,231946,2,Entire home/apt,Entire condo,2,1.0,1.0,1 bath,162.0,2,365,False,0363 22DC 0E52 B70B 0FB8
9,53921,252245,10,Entire home/apt,Entire rental unit,3,1.0,NaN,1 bath,NaN,1,21,False,0363 B43C B1D4 2666 3739



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False
5,443406859,cfe69dd56cffef559069b30216f8954b57e1de886b92a3...,False
6,2626085,43c4cdd7f1413364dd809c6c92c20baed35faa51f84e76...,False
7,74999205,1033d2369452b0969c1f63061af401f4581a92873b5659...,True
8,7969106,1b41831025e74b04a73fe88e99233d09f39660eac85bf6...,False
9,6127483,8cffc835ea5e24b102cf446e9d369edad9f3a2bf91bfbc...,True



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart
5,6,Bijlmer-Centrum
6,7,Geuzenveld - Slotermeer
7,8,Buitenveldert - Zuidas
8,9,Noord-West
9,10,IJburg - Zeeburgereiland



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,1139703219883851440,54255244,2024-04-21,188921026,a64787088825a01dfc7e9a09cb3654ce4e0ad0743661e2...,397
1,1145582279735733326,54255244,2024-04-29,90469529,1a3cc91ea2ba7481251ceb393bad69a38fb81bf095ec4c...,274
2,1167322314649755882,54255244,2024-05-29,355624981,6fbe3455de0e36efeaa8117e7fdce830fb56e478019963...,27
3,1188349400045522534,54255244,2024-06-27,60806021,a744bebae7f73bc9dab3c74641ed20a0596767c2302927...,259
4,1196282675497238794,54255244,2024-07-08,19579748,9cce8548d18031519b43418d6473d025b063ad605217cd...,486
5,604810597153836614,54271157,2022-04-14,9227748,7a4b74b7fddfb0a561c8d54a351f3781a94a73a9e49d55...,367
6,607793169219181420,54271157,2022-04-18,199390130,dcf2886eecd42b1ebaceb2ee0d5d753f13de53987aebff...,113
7,614983679556766956,54271157,2022-04-28,112457760,cb74fb419edb14cd6e1511d7a998dbbafba2af01f8fb81...,345
8,618642193319418014,54271157,2022-05-03,449230789,1a3d2570c91240f57f6b7358505a6037b413d2282dee9c...,334
9,625115040253965175,54271157,2022-05-12,25900607,c0136bafc9b84171d5af70d6f192f013fafc6085bec4c0...,291



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,703359952526896727,2026-07-18,False,None,None,4,21
1,703359952526896727,2026-07-19,False,None,None,4,21
2,703359952526896727,2026-07-20,False,None,None,4,21
3,703359952526896727,2026-07-21,False,None,None,4,21
4,703359952526896727,2026-07-22,False,None,None,4,21
5,703359952526896727,2026-07-23,False,None,None,4,21
6,703359952526896727,2026-07-24,False,None,None,4,21
7,703359952526896727,2026-07-25,False,None,None,4,21
8,703359952526896727,2026-07-26,False,None,None,4,21
9,703359952526896727,2026-07-27,False,None,None,4,21


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [105]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [106]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

# TODO:
# 1. Compute earliest valid cutoff using calendar_min_date and PAST_WINDOW_DAYS.
# 2. Compute latest valid cutoff using calendar_max_date and FUTURE_WINDOW_DAYS.
# 3. Choose a valid cutoff_date.
# 4. Compute history_start_date and label_end_date.
#
# Hint:
# earliest valid cutoff = calendar_min_date + PAST_WINDOW_DAYS
# latest valid cutoff = calendar_max_date - FUTURE_WINDOW_DAYS

earliest_cutoff_allowed_by_calendar = calendar_min_date + pd.DateOffset(PAST_WINDOW_DAYS)  # TODO
latest_cutoff_allowed_by_calendar = calendar_max_date - pd.DateOffset(FUTURE_WINDOW_DAYS)    # TODO

cutoff_date = pd.to_datetime('2026-01-01')          # TODO
history_start_date = cutoff_date - pd.DateOffset(PAST_WINDOW_DAYS)   # TODO
label_end_date = cutoff_date + pd.DateOffset(FUTURE_WINDOW_DAYS)   # TODO

# TODO: print all dates and assert that the windows are valid.
print( 
    f"""earliest_cutoff_allowed_by_calendar =  {earliest_cutoff_allowed_by_calendar} ,
    latest_cutoff_allowed_by_calendar = {latest_cutoff_allowed_by_calendar},
    cutoff_date = {cutoff_date},
    history_start_date = {history_start_date},
    label_end_date = {label_end_date}"""
)

assert PAST_WINDOW_DAYS> 0 , f"number greater than 0 expected, got:{PAST_WINDOW_DAYS} "
assert FUTURE_WINDOW_DAYS> 0, f"number greater than 0 expected, got:{FUTURE_WINDOW_DAYS} "

if  earliest_cutoff_allowed_by_calendar is None or latest_cutoff_allowed_by_calendar is None or cutoff_date is None :
    raise NotImplementedError("TODO: implement cutoff-date logic." )

earliest_cutoff_allowed_by_calendar =  2025-12-10 00:00:00 ,
    latest_cutoff_allowed_by_calendar = 2026-08-11 00:00:00,
    cutoff_date = 2026-01-01 00:00:00,
    history_start_date = 2025-10-03 00:00:00,
    label_end_date = 2026-01-31 00:00:00


## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [107]:
# TODO: complete the PII audit table.
# Add rows for all sensitive or identity-linking columns you find relevant.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    # TODO: add host_id
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "host identifier",
        "decision": "keep as entity key only",
        "reason": "not a model input"
    },# TODO: add host_pseudo_id
    {   
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "pseudonimized id",
        "decision": "remove",
        "reason": "not a model input"
    },    # TODO: add license
    {
        "table": "listing",
        "column": "license",
        "pii_type": "sensitive data",
        "decision": "remove",
        "reason": "not a model input"
    },    # TODO: add review_id
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "entity identifier",
        "decision": "remove",
        "reason": "not a model input"
    },# TODO: add reviewer_id
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "entity identifier",
        "decision": "remove",
        "reason": "not a model input"
    },# TODO: add reviewer_pseudo_id
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "pseudonimized id",
        "decision": "remove",
        "reason": "not a model input"
    }
    
    
])

pii_audit

,table,column,pii_type,decision,reason
0,listing,listing_id,entity identifier,keep as entity key only,needed to define one row per listing; not a mo...
1,listing,host_id,host identifier,keep as entity key only,not a model input
2,host,host_pseudo_id,pseudonimized id,remove,not a model input
3,listing,license,sensitive data,remove,not a model input
4,review,review_id,entity identifier,remove,not a model input
5,review,reviewer_id,entity identifier,remove,not a model input
6,review,reviewer_pseudo_id,pseudonimized id,remove,not a model input


## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [108]:
# TODO: write SQL to load the required listing columns.
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    room_type,
    property_type,
    accommodates,
    bedrooms,
    beds,
    bathrooms_text,
    listing_price,
    minimum_nights,
    maximum_nights,
    instant_bookable

FROM core.listing;
""")

# TODO: write SQL to load host columns.
host_df = read_sql("""
SELECT
    host_id,
    is_superhost as host_is_superhost
    
FROM core.host;
""")

# TODO: write SQL to load neighbourhood columns.
neighbourhood_df = read_sql("""
SELECT
    neighbourhood_id,
    name as neighbourhood_name 
    
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

listing: (10480, 13)
host: (9201, 2)
neighbourhood: (22, 2)


## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [109]:
# TODO: normalize boolean columns.
# Example target columns:
listing_df["instant_bookable"] = listing_df["instant_bookable"].astype(int)
host_df["host_is_superhost"] = host_df["host_is_superhost"].astype("boolean").astype("Int64")



# TODO: normalize numeric listing columns.
numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    pd.to_numeric(listing_df[col])   
    pass

import re

def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan

    text = str(text).strip().lower()

    if text == "half-bath":
        return 0.5

    match = re.match(r"(\d+(\.\d+)?)", text)
    if match:
        value = float(match.group(1))

        if "shared" in text:
            return value * 0.2
        if "private" in text:
            return value * 2
        

        return value

    return np.nan
    
    # TODO: implement parser.
    #raise NotImplementedError("TODO: implement parse_bathrooms")


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)    
#listing_df[["bathrooms_text", "bathrooms"]].head(10)
listing_df = listing_df.drop(columns="bathrooms_text")


## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [110]:
host_listing_merged = listing_df.merge(host_df, on="host_id")
host_listing_features = (host_listing_merged).groupby(["host_id"], as_index=False).size().rename(columns={
    "size": "host_listing_count"}).merge(host_df, on="host_id").drop_duplicates()
host_listing_features

,host_id,host_listing_count,host_is_superhost
0,1662,1,0
1,3592,1,0
2,14589,1,0
3,42599,1,0
4,57722,1,0
...,...,...,...
9196,714363048,1,0
9197,715439809,1,0
9198,715849738,1,0
9199,716142178,1,0


In [111]:
# TODO: create host_listing_features with one row per host_id.
# x = """
# SELECT
#     h.host_id,
#     h.is_superhost,
#     COUNT(*) AS host_listing_count
# FROM listing l
# JOIN host h
#     ON h.host_id = l.host_id
# GROUP BY h.host_id, h.is_superhost;
# """
# host_listing_features = read_sql(x)

host_listing_merged = listing_df.merge(host_df, on="host_id")
host_listing_features = (host_listing_merged).groupby(["host_id"], as_index=False).size().rename(columns={
    "size": "host_listing_count"}).drop_duplicates()

assert host_listing_features["host_id"].duplicated().sum() == 0


# TODO: merge listing, host, host_listing_features, and neighbourhood.
base_listing_features = (listing_df
             .merge(host_df, on="host_id", how="left")
             .merge(host_listing_features, on="host_id", how="left")
             .merge(neighbourhood_df, on="neighbourhood_id", how="left")) 

base_listing_features


# TODO: choose privacy-safe static feature columns.
static_feature_cols = [
    "listing_id",
    "neighbourhood_name",
    "host_is_superhost",
    "host_listing_count",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable"

]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()
#host_listing_features
#base_listing_features
#merged_df

#host_listing_features

(10480, 14)


,listing_id,neighbourhood_name,host_is_superhost,host_listing_count,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,instant_bookable
0,27886,Centrum-West,1,1,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,3,356,0
1,28871,Centrum-West,1,2,Private room,Private room in rental unit,2,1.0,1.0,0.2,89.0,2,730,0
2,29051,Centrum-Oost,1,2,Private room,Private room in condo,2,1.0,1.0,0.2,61.0,2,730,0
3,44391,Centrum-Oost,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,730,0
4,48373,Buitenveldert - Zuidas,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,1125,0


## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [112]:
#review_id	listing_id	review_date	reviewer_id	reviewer_pseudo_id	comment_len
# TODO: write SQL aggregation over core.review.
# Use CAST(:cutoff_date AS date) when using the cutoff inside SQL.
#-- TODO: aggregate review features
review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_id) AS unique_reviewers_before_cutoff,
        AVG(comment_len) AS avg_comment_len_before_cutoff,
        MAX(comment_len) AS max_comment_len_before_cutoff,
        (CAST(:cutoff_date AS date) - MAX(review_date)) AS days_since_last_review
        
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

# TODO: convert feature columns to numeric where needed.
for col in review_features.columns:
    review_features[col] = pd.to_numeric(review_features[col])

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()


(9383, 6)


,listing_id,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review
0,27886,311,311,302.167203,1917,116
1,28871,732,729,201.236339,1265,116
2,29051,849,841,245.108363,2253,115
3,44391,42,42,242.309524,891,1230
4,48373,5,5,272.200000,949,613


## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [113]:
# TODO: write SQL aggregation over core.calendar_day.
# Use history_start_date and cutoff_date.
#
# Required output examples:
# - available_days_last_90d
# - available_rate_last_90d
# - avg_minimum_nights_calendar_last_90d
# - avg_maximum_nights_calendar_last_90d
# - available_days_last_30d
# - available_rate_last_30d
# - avg_minimum_nights_calendar_last_30d
# - avg_maximum_nights_calendar_last_30d
#-- TODO: aggregate calendar history features


calendar_features_all = read_sql(
    """
    SELECT
        listing_id,

        COUNT(CASE WHEN available = TRUE THEN 1 END) AS available_days_last_90d,

        AVG(CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END)
            AS available_rate_last_90d,

        AVG(minimum_nights) AS avg_minimum_nights_calendar_last_90d,

        AVG(maximum_nights) AS avg_maximum_nights_calendar_last_90d,

        /* -------------------- 30D WINDOW -------------------- */

        COUNT(CASE
                WHEN available = TRUE
                 AND date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                THEN 1
             END) AS available_days_last_30d,

        AVG(CASE
                WHEN date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                THEN CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END
            END) AS available_rate_last_30d,

        AVG(CASE
                WHEN date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                THEN minimum_nights
            END) AS avg_minimum_nights_calendar_last_30d,

        AVG(CASE
                WHEN date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                THEN maximum_nights
            END) AS avg_maximum_nights_calendar_last_30d

    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date < CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    }
)

# TODO: convert numeric columns.
for col in calendar_features_all.columns:
    calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")



assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

(10480, 9)


,listing_id,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d
0,27886,17,0.188889,3.000000,30.0,1,0.033333,3.000000,30.0
1,28871,15,0.166667,1.888889,730.0,6,0.200000,1.933333,730.0
2,29051,13,0.144444,1.977778,730.0,2,0.066667,2.000000,730.0
3,44391,0,0.000000,3.000000,730.0,0,0.000000,3.000000,730.0
4,48373,0,0.000000,3.000000,1125.0,0,0.000000,3.000000,1125.0


## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [114]:
# TODO: write SQL to build one label row per listing.
# Use only dates after cutoff_date and up to label_end_date.
# -- TODO: future_calendar_days_observed_30d
# -- TODO: future_available_days_30d
# -- TODO: future_available_rate_30d
# -- TODO: high_demand_proxy

label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS future_calendar_days_observed_30d,
        COUNT(CASE WHEN available = TRUE THEN 1 END) AS future_available_days_30d,
        AVG(CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END) AS future_available_rate_30d,
        CASE
            WHEN AVG(CASE WHEN available = TRUE THEN 1.0 ELSE 0.0 END) <= 0.30 
            then 1
            ELSE 0.0 
        END as high_demand_proxy
    
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    }
)

# TODO: convert numeric columns and make high_demand_proxy integer.

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

(10480, 5)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy
0,1443670960781261954,30,0,0.000000,1.0
1,896043282611946316,30,0,0.000000,1.0
2,39969190,30,0,0.000000,1.0
3,958726726744532841,30,0,0.000000,1.0
4,1476051889347548382,30,28,0.933333,0.0


In [115]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

,high_demand_proxy,count,percentage
0,1.0,7141,0.6814
1,0.0,3339,0.3186


## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [116]:
# TODO: join static_features, review_features, calendar_features_all, and label_df.

feature_df = (static_features
          .merge(review_features, on="listing_id", how="inner")
          .merge(calendar_features_all, on="listing_id", how="inner")
          .merge(label_df, on="listing_id", how="inner")

)
# TODO: add cutoff_date and dataset_version columns.

feature_df['cutoff_date'] = cutoff_date
feature_df['dataset_version'] = DATASET_VERSION

# TODO: handle missing days_since_last_review for listings with no reviews.

feature_df['days_since_last_review'] = feature_df['days_since_last_review'].fillna(250)

# TODO: fill missing review count features with zero.
review_features_cols = ['total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 
                        'avg_comment_len_before_cutoff']

for col in review_features_cols:
    feature_df[review_features_cols] = feature_df[review_features_cols].fillna(0)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()
#feature_df.columns.duplicated().any()


(9383, 33)


,listing_id,neighbourhood_name,host_is_superhost,host_listing_count,room_type,property_type,accommodates,bedrooms,beds,bathrooms,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Centrum-West,1,1,Private room,Private room in houseboat,2,1.0,1.0,1.5,...,1,0.033333,3.000000,30.0,30,0,0.000000,1.0,2026-01-01,v1_student
1,28871,Centrum-West,1,2,Private room,Private room in rental unit,2,1.0,1.0,0.2,...,6,0.200000,1.933333,730.0,30,1,0.033333,1.0,2026-01-01,v1_student
2,29051,Centrum-Oost,1,2,Private room,Private room in condo,2,1.0,1.0,0.2,...,2,0.066667,2.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
3,44391,Centrum-Oost,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,...,0,0.000000,3.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
4,48373,Buitenveldert - Zuidas,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,...,0,0.000000,3.000000,1125.0,30,0,0.000000,1.0,2026-01-01,v1_student


## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [117]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

# TODO: compute missing rates.
missing_rates = feature_df.isna().mean().head(30)

# TODO: find high-missing columns outside protected columns.
high_missing_cols = missing_rates[
    (missing_rates > HIGH_MISSING_DROP_THRESHOLD) &
    (missing_rates.index.isin(protected_columns))
].index.to_list()

# TODO: find constant columns outside protected columns.
constant_cols = [
    col for col in feature_df.columns
    if col not in protected_columns
    and feature_df[col].nunique(dropna=False) == 1 
        
]

#TODO: drop them from feature_df.

columns_to_drop = high_missing_cols  # TODO

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

Columns to drop: []
New shape: (9383, 33)


## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [118]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
assert duplicate_count == 0
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
assert missing_target_count == 0
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())
assert unique_target_values == [0.0, 1.0]

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))
assert not present_forbidden_columns 

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]
assert not future_leakage_columns

# TODO: add asserts for each validation rule.
# for example:
# assert duplicate_count == 0

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))


duplicate_count: 0
missing_target_count: 0
unique_target_values: [0.0, 1.0]
present_forbidden_columns: []
future_leakage_columns: []
model_input_column_count: 26


In [119]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

,column,missing_rate
0,listing_price,0.443888
1,beds,0.441223
2,bedrooms,0.029948
3,host_is_superhost,0.011723
4,bathrooms,0.002132
5,listing_id,0.000000
6,host_listing_count,0.000000
7,room_type,0.000000
8,neighbourhood_name,0.000000
9,property_type,0.000000


,high_demand_proxy,count,percentage
0,1.0,7141,0.6814
1,0.0,3339,0.3186


,index,future_calendar_days_observed_30d
0,count,9383.0
1,mean,30.0
2,std,0.0
3,min,30.0
4,25%,30.0
5,50%,30.0
6,75%,30.0
7,max,30.0


## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [120]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

# TODO: build metadata dictionary.
metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff" : str(cutoff_date),
    "PAST_WINDOW_DAYS": PAST_WINDOW_DAYS,
    "FUTURE_WINDOW_DAYS" : FUTURE_WINDOW_DAYS,
    "target" : "high_demand_proxy",
    "exclusion_rules" : "forbidden_columns",
    "target_definition" : "The target is to predict if the listing is a high demad proxy"
    
    # TODO: add cutoff, windows, source tables, target definition, exclusion rules
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# TODO: build validation_report dictionary.
validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "missing_report" : missing_report.to_dict(),
    "label_distribution" : label_distribution.to_dict(),
    "calendar_coverage_summary" : calendar_coverage_summary.to_dict()
    # TODO: add missing_report, label_distribution, calendar_coverage_summary
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

Saved CSV: c:\Users\lenovo\Desktop\hw2\1\data\features\listing_availability_features_v1_student.csv
Saved Parquet: c:\Users\lenovo\Desktop\hw2\1\data\features\listing_availability_features_v1_student.parquet
Saved metadata: c:\Users\lenovo\Desktop\hw2\1\data\features\listing_availability_features_v1_student_metadata.json
Saved validation report: c:\Users\lenovo\Desktop\hw2\1\data\features\listing_availability_features_v1_student_validation_report.json
Saved PII audit: c:\Users\lenovo\Desktop\hw2\1\data\features\pii_audit_v1_student.csv


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [121]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()

Final shape: (9383, 33)


,listing_id,neighbourhood_name,host_is_superhost,host_listing_count,room_type,property_type,accommodates,bedrooms,beds,bathrooms,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Centrum-West,1,1,Private room,Private room in houseboat,2,1.0,1.0,1.5,...,1,0.033333,3.000000,30.0,30,0,0.000000,1.0,2026-01-01,v1_student
1,28871,Centrum-West,1,2,Private room,Private room in rental unit,2,1.0,1.0,0.2,...,6,0.200000,1.933333,730.0,30,1,0.033333,1.0,2026-01-01,v1_student
2,29051,Centrum-Oost,1,2,Private room,Private room in condo,2,1.0,1.0,0.2,...,2,0.066667,2.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
3,44391,Centrum-Oost,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,...,0,0.000000,3.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
4,48373,Buitenveldert - Zuidas,0,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,...,0,0.000000,3.000000,1125.0,30,0,0.000000,1.0,2026-01-01,v1_student


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9383 entries, 0 to 9382
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   listing_id                            9383 non-null   int64         
 1   neighbourhood_name                    9383 non-null   object        
 2   host_is_superhost                     9273 non-null   Int64         
 3   host_listing_count                    9383 non-null   int64         
 4   room_type                             9383 non-null   object        
 5   property_type                         9383 non-null   object        
 6   accommodates                          9383 non-null   int64         
 7   bedrooms                              9102 non-null   float64       
 8   beds                                  5243 non-null   float64       
 9   bathrooms                             9363 non-null   float64       
 10  